<a href="https://colab.research.google.com/github/ricvazquez/ColabFiles/blob/main/Sesion12_Evaluacion_Datos_Categoricos_273509.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Programación para Analítica Descriptiva y Predictiva**
**Maestría en Inteligencia Artificial y Analítica de Datos**

# Sesión 12: Evaluación — Limpieza y Transformación de Datos Categóricos

**Entrega individual**

- **Nombre**: Ricardo Vazquez Macias
- **Matrícula** 273509

Esta evaluación aplica los tres temas de la Sesión 12 (errores tipográficos y valores inconsistentes, alta cardinalidad, tipos incorrectos) a un dataset que no se trabajó en clase: **Telco Customer Churn**.


No hay una única respuesta correcta en varias de las actividades — lo que se evalúa es que la conclusión esté respaldada por el código que la sustenta, no solo la conclusión en sí.

## Preparación

In [ ]:
import pandas as pd
import numpy as np

url = 'https://raw.githubusercontent.com/treselle-systems/customer_churn_analysis/master/WA_Fn-UseC_-Telco-Customer-Churn.csv'
df = pd.read_csv(url)
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


---
## Actividad 1 — Formato y valores inconsistentes (15 pts)

Revisa **todas** las columnas categóricas del dataset (no elijas solo una) en busca de variantes de formato (mayúsculas, espacios) que deberían normalizarse.

In [ ]:
# 1.1 — Recorre todas las columnas categóricas con .value_counts() o .unique()
# para inspeccionar sus valores.

# Hacemos un ciclo for para recorrer todo el dataset cuyas columnas sean object
for i in df.select_dtypes(include='object').columns:
  # eh imprimimos son valores unicos y cuantas veces se repiten
  print(df[i].value_counts())
  print('----------------------------------')


customerID
3186-AJIEK    1
7590-VHVEG    1
5575-GNVDE    1
8775-CEBBJ    1
2823-LKABH    1
             ..
6713-OKOMC    1
1452-KIOVK    1
9305-CDSKC    1
9237-HQITU    1
7795-CFOCW    1
Name: count, Length: 7043, dtype: int64
----------------------------------
gender
Male      3555
Female    3488
Name: count, dtype: int64
----------------------------------
Partner
No     3641
Yes    3402
Name: count, dtype: int64
----------------------------------
Dependents
No     4933
Yes    2110
Name: count, dtype: int64
----------------------------------
PhoneService
Yes    6361
No      682
Name: count, dtype: int64
----------------------------------
MultipleLines
No                  3390
Yes                 2971
No phone service     682
Name: count, dtype: int64
----------------------------------
InternetService
Fiber optic    3096
DSL            2421
No             1526
Name: count, dtype: int64
----------------------------------
OnlineSecurity
No                     3498
Yes                    

**1.2 — Conclusión (responde aquí en Markdown):**

¿Encontraste alguna columna con inconsistencias de formato? Si sí, ¿cuál y qué código usarías para corregirla? Si no encontraste ninguna, dilo explícitamente — es una conclusión válida siempre que esté respaldada por lo que revisaste en 1.1.

_Tu respuesta: Basado en el analisis anterior, se puede ver que muchas de las columnas las respuestas cumplen con el valor correcto, pareciera que las columnas ya fueron normalizadas

---
## Actividad 2 — Valores inválidos (20 pts)

Varias columnas de este dataset (`OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`) tienen un tercer valor además de `'Yes'`/`'No'`: `'No internet service'`. De forma similar, `MultipleLines` tiene `'No phone service'`.

In [ ]:
df['OnlineSecurity'].value_counts()

,count
OnlineSecurity,
No,3498
Yes,2019
No internet service,1526


In [ ]:
# 2.1 — Verifica: ¿las filas con 'No internet service' en OnlineSecurity
# coinciden con las filas donde InternetService == 'No'?
# (pista: cruza ambas columnas con pd.crosstab o filtrando)

#Con la funcion cross tab cruzamos las columnas para ver elnumero de filas y los valores
print(pd.crosstab(df['OnlineSecurity'], df['InternetService']))

# Confirmamos sacado los sum y comparandolos
totOn = (df['OnlineSecurity'] == 'No internet service').sum()
totInt = (df['InternetService'] == 'No').sum()
print('Online security sin internet:',totOn)
print('Sin internet service:',totInt)



InternetService       DSL  Fiber optic    No
OnlineSecurity                              
No                   1241         2257     0
No internet service     0            0  1526
Yes                  1180          839     0
Online security sin internet: 1526
Sin internet service: 1526


**2.2 — Conclusión (responde aquí en Markdown):**

¿`'No internet service'` es un valor inválido (como `Absurd`/`YOLO` en la sesión de clase) o es una categoría legítima? Justifica tu respuesta con lo que verificaste en 2.1. ¿Tomarías alguna acción sobre esta columna, o la dejarías tal cual?

_Tu respuesta:_ Es un respuesta legitima, ya que, si es un categoria al haber gente que no cuenta con el servicio de internet no se puede categorizar dentro de las otras dos

---
## Actividad 3 — Alta cardinalidad (25 pts)

In [ ]:
# 3.1 — Calcula .nunique() para TODAS las columnas del dataset (no solo las categóricas)
# y la razón (valores únicos / total de filas) para cada una.

# Recorremos columna x  y sacamos los unique y con eso l dividimos en el len de df
for i in df.columns:
  val = df[i].nunique()
  rate = val/len(df)
  print('Columna:',i,'Valores unicos:',val,' ',round(rate*100,2),'% del total de filas')
  print('----------------------------------')

Columna: customerID Valores unicos: 7043   100.0 % del total de filas
----------------------------------
Columna: gender Valores unicos: 2   0.03 % del total de filas
----------------------------------
Columna: SeniorCitizen Valores unicos: 2   0.03 % del total de filas
----------------------------------
Columna: Partner Valores unicos: 2   0.03 % del total de filas
----------------------------------
Columna: Dependents Valores unicos: 2   0.03 % del total de filas
----------------------------------
Columna: tenure Valores unicos: 73   1.04 % del total de filas
----------------------------------
Columna: PhoneService Valores unicos: 2   0.03 % del total de filas
----------------------------------
Columna: MultipleLines Valores unicos: 3   0.04 % del total de filas
----------------------------------
Columna: InternetService Valores unicos: 3   0.04 % del total de filas
----------------------------------
Columna: OnlineSecurity Valores unicos: 3   0.04 % del total de filas
--------------

**3.2 — Conclusión (responde aquí en Markdown):**

¿Qué columna(s) tienen alta cardinalidad? Para la columna con mayor cardinalidad: ¿por qué nunca deberías usarla como variable predictora en un modelo, incluso si la codificaras? (relaciona tu respuesta con lo discutido en clase sobre identificadores únicos)

_Tu respuesta:_ Quitando la columna de ID (CustomerID), la que le sigue es Total charges la cual tiene una cardinalidad del 92.1%, y esta no seriviria de nada como variable predictora, ya que no se puede agrupar, o si se agrupa igual no simplificaria nada

**3.3 — Agrupación "Top 10 + Otros"**

En clase agrupaste `country` de Netflix Titles en sus 10 categorías más frecuentes + `'Otros'`, reduciendo su cardinalidad. Aplica la misma técnica aquí sobre la columna de mayor cardinalidad que identificaste en 3.1 (pista: `.value_counts().head(10)`, luego `.where()` + `.isin()`, igual que en el notebook de clase).

In [ ]:
# Aplica el agrupamiento Top 10 + Otros sobre la columna de mayor cardinalidad

# Primero creamos la lista del top 10 con la ayuda de index to list
topTotalCh=df['TotalCharges'].value_counts().head(10).index.tolist()
print(topTotalCh)
# Y ahora para no modificar la columna original, creamos una nueva y en ella guardamos el top 10 y los que estan fuera los renombramos como 'Otros'
df['TotalCharges_ag'] = df['TotalCharges'].where(df['TotalCharges'].isin(topTotalCh), 'Otros')
df['TotalCharges_ag'].value_counts()

[' ', '20.2', '19.75', '20.05', '19.9', '19.65', '19.55', '45.3', '20.15', '20.25']


,count
TotalCharges_ag,
Otros,6962
20.2,11
,11
19.75,9
19.65,8
20.05,8
19.9,8
19.55,7
45.3,7


**3.4 — Conclusión (responde aquí en Markdown):**

Después de agrupar, ¿la columna resultante te parece útil para un modelo? Compara este caso con el de `country` en el notebook de clase: ¿por qué agrupar en "Top 10 + Otros" funciona bien para una variable como `country`, pero no resuelve el problema real de la columna que agrupaste aquí?

_Tu respuesta:_No me parece util la columna resultante, ya que 'otros' engloba mas del 90% del dataset, a diferencia de countrys vista en el curso donde otros no era mas de la mitad del dataset, y los datos agrupados tampoco pudieran servir de mucho

---
## Actividad 4 — Tipos de dato (30 pts)

In [ ]:
df['TotalCharges'].dtype

dtype('O')

**4.1 — Investiga (responde en Markdown):**

`TotalCharges` contiene valores numéricos (montos en dólares), pero pandas la cargó como `object`, no como `float`. Investiga por qué — revisa si hay algún valor que no se vea como un número normal.

_Tu respuesta:_ Esto se debe a que pudiera contener datos en cadena de texto vacias como " " los cuales pandas pudiera detectar como texto

In [ ]:
# 4.2 — Corrige el tipo de TotalCharges.
# Pista: pd.to_numeric() con el parámetro errors= te puede ayudar a identificar
# o manejar los valores problemáticos que encontraste en 4.1.

# Convertimos la columna a float
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].dtype

dtype('float64')

In [ ]:
# 4.3 — Convierte a category las columnas categóricas que, según lo que calculaste
# en la Actividad 3, tengan cardinalidad baja y valores fijos.
# Verifica con .dtypes que el cambio se aplicó correctamente.


for i in df.select_dtypes(include='object').columns:
  if df[i].nunique() < 10:
    df[i] = df[i].astype('category')
df.dtypes

,0
customerID,object
gender,category
SeniorCitizen,int64
Partner,category
Dependents,category
tenure,int64
PhoneService,category
MultipleLines,category
InternetService,category
OnlineSecurity,category


---
## Reflexión final (10 pts)

Con base en las 4 actividades anteriores, responde:

1. De las alertas que detectaste (formato, valores inválidos, cardinalidad, tipos), ¿cuál te pareció más fácil de decidir y cuál más difícil? ¿Por qué?

La mas sencilla de decidir fue la de cardinalidad porque en este caso,el porcentaje de cardinalidad de las columnas que no aplicaban era muy alto, y de las que si aplicaban era muy bajo, haciendo muy sencillo poder decidir en cuales aplicaba y cuales no. Y la dificil pudiera ser la de valores invalidos, ya que practicamente no tenia valores invalidos, pero por creer que si, le tuve que dar varias vueltas al dataset


2. Si tuvieras que entregar este dataset ya "perfilado" a un compañero para que construya un modelo predictivo, ¿qué le dirías sobre `customerID` y sobre `TotalCharges`?

Le diria que ambas columnas no se pueden agrupar, que son datos con una alta cardinalidad, y que inclusp no seriviria de nada desgastarse agrupandolos, lo mejor seria usarlos tal y como estan
_Tu respuesta:_

---
## Rúbrica de evaluación

| Actividad | Puntos | Criterio |
|---|---|---|
| 1. Formato y valores inconsistentes | 15 | Revisó todas las columnas categóricas (no solo una); código comentado y conclusión (1.2) respaldada por lo que se observó, no solo afirmada |
| 2. Valores inválidos | 20 | Verificó la relación entre columnas antes de concluir; la conclusión (2.2) justifica con evidencia, no solo con intuición |
| 3. Alta cardinalidad | 25 | Calcula `.nunique()` para todas las columnas; aplica correctamente el agrupamiento Top 10 + Otros; la conclusión (3.4) explica por qué agrupar no resuelve el problema de un identificador único |
| 4. Tipos de dato | 30 | Identifica la causa raíz del `dtype` incorrecto (4.1); corrige `TotalCharges` sin perder información; conversión a `category` justificada por cardinalidad, no aplicada al azar |
| Reflexión final | 10 | Conecta las 4 actividades entre sí; no es una respuesta genérica o intercambiable con cualquier dataset |
| **Total** | **100** | |

**Nota sobre las conclusiones:** cada actividad tiene su propia pregunta de conclusión (1.2, 2.2, 3.4, 4.1) — esas respuestas se califican como parte de la actividad correspondiente, no solo la Reflexión final. Comenta tu código donde tomes una decisión (por ejemplo, por qué elegiste cierto umbral o cierta corrección) — el comentario también cuenta dentro del puntaje de cada actividad.